# Cell type phenotyping and annotation

Explore the full pancreas cohort using the updated Islet Explorer protein assignments. The same UMAP shows where cell identities, disease groups, and protein signals overlap. The heatmap and dotplots summarize the proteins that distinguish each phenotype.

Adapted from the [original PhenoCycler analysis notebook](https://github.com/smith6jt-cop/Phenocycler_Analysis/blob/ee170f13ebb892bc395bb289625df038ad0fadb1/notebooks/02_phenotyping.ipynb).

## 1. Load the processed PhenoCycler data

The notebooks require the November 13 Islet Explorer protein release that excludes EpCAM from typing and rebuilds the pancreas UMAP without that marker. The load step checks the release ID and the marker list saved with the embedding. The cohort UMAP sample enriches uncommon cell types. Marker summaries and cell proportions use all admitted cells.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import TwoSlopeNorm
import anndata as ad
import scanpy as sc
from IPython.display import display

# From a separate checkout, set PHENOCYCLER_DATA to the folder containing the .h5ad files.
cwd = Path.cwd()
candidates = [Path(os.environ["PHENOCYCLER_DATA"])] if "PHENOCYCLER_DATA" in os.environ else [
    cwd / "analysis" / "phenocycler" / "scanpy", cwd / "phenocycler" / "scanpy",
    cwd / "scanpy", cwd.parent / "scanpy", cwd]
data = next((d for d in candidates if (d / "typing_release.json").is_file()), None)
if data is None:
    raise FileNotFoundError("Set PHENOCYCLER_DATA to the no-EpCAM Scanpy release folder containing typing_release.json.")
sc.set_figure_params(dpi=100, facecolor="white", fontsize=10, frameon=False)
plt.rcParams.update({"axes.spines.top":False, "axes.spines.right":False})

# Locate the shared plotting module when running from a notebook, its parent, or the Senior checkout.
import sys
helper_paths = [cwd, cwd/"notebooks", cwd/"phenocycler"/"notebooks", cwd/"analysis"/"phenocycler"/"notebooks",
                data.parent/"notebooks"]
helper_dir = next((p for p in helper_paths if (p/"phenotype_signatures.py").is_file()), None)
if helper_dir is None:
    raise FileNotFoundError("Run from the Senior checkout or its analysis/phenocycler/notebooks directory.")
sys.path.insert(0, str(helper_dir))
from phenotype_signatures import (PAN_STRUCTURE, HORMONES, IMMUNE, T_CELL, DISPLAY_NAMES,
                                  donor_heatmap, show_profiles, conflict_summary, validate_release)
from phenotype_signatures import (cell_subtypes, phenotype_colors, phenotype_key, names_in_order, subtype_display_name)
from phenotype_categories import category_panels, draw_identity_map
release = validate_release(data)

DONOR = "115"                      # Any donor listed in the manifest
DISEASE = "ND"                     # ND, AAb+, or T1D
LEVEL = "subtype"                   # Marker profiles: subtype, primary, detail, immune_class, t_subtype, macrophage_profile
UMAP_MARKERS = ["GCG", "INS", "SST", "Podoplanin", "B3TUBB", "CD56", "CD31", "Ker8_18", "CD3e"]
PAIR = ("INS","SST")
group_order = ["ND","AAb+","T1D"]
group_ink = {"ND":"#23699b","AAb+":"#ab6c06","T1D":"#ae4643"}
counts = pd.read_parquet(data / "donor_counts.parquet").query("tissue == 'Pancreas'").copy()
evidence = pd.read_parquet(data / "donor_markers.parquet").query("tissue == 'Pancreas'").copy()
counts["donor"] = counts.donor.astype(str)
evidence["donor"] = evidence.donor.astype(str)
manifest = pd.read_csv(data / "pancreas_manifest.csv",dtype={"donor":str})
if DONOR not in set(manifest.donor) or DISEASE not in group_order:
    raise ValueError("Choose a donor and group from pancreas_manifest.csv")
atlas = sc.read_h5ad(data / "pancreas_umap.h5ad")
coords = np.asarray(atlas.obsm["X_umap"])
mapped = atlas[np.isfinite(coords).all(axis=1)].copy()
print(f"{len(manifest)} pancreas donors · {manifest.cells.sum():,} admitted cells · {mapped.n_obs:,} displayed on the joint UMAP")
print("Disease groups:", ", ".join(f"{g} ({n} donors)" for g,n in manifest.groupby("disease").size().items()))
inks = phenotype_colors(counts, "Pancreas")
mapped.obs["subtype"] = cell_subtypes(mapped.obs, "Pancreas")

## 2. Visualize cell subtypes on the shared protein UMAP

The cohort, disease group and donor share coordinates, axis limits and subtype colors. Each cell takes its most specific recorded assignment, including its macrophage profile when available.

The identity and assignment-QC figures use the same saved coordinates. The disease overview includes all mapped cells.

In [ ]:
limits = [(coords[np.isfinite(coords[:,k]),k].min(), coords[np.isfinite(coords[:,k]),k].max()) for k in (0,1)]
views = [(mapped, "All donors"), (mapped[mapped.obs.disease.astype(str).eq(DISEASE)], DISEASE),
         (mapped[mapped.obs.donor.astype(str).eq(DONOR)], f"Donor {DONOR}")]
fig, ax = plt.subplots(figsize=(5.5,4.5), layout="constrained")
sc.pl.umap(mapped, color="disease", palette=group_ink, size=3, ax=ax,
           title="Disease group · all mapped cells", legend_loc="none", show=False)
handles = [Line2D([], [], marker="o", linestyle="", markersize=5, color=group_ink[g], label=g)
           for g in group_order]
ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(.5,0), ncol=3, frameon=False, fontsize=8)
ax.set(xlim=limits[0], ylim=limits[1], aspect="equal")
plt.show()

all_labels = names_in_order(mapped.obs.subtype.unique(), "Pancreas")
for panel, labels in category_panels(all_labels, "Pancreas"):
    legend_height = .24 * np.ceil(len(labels)/3) + .3
    fig = plt.figure(figsize=(14, 4.8+legend_height), layout="constrained")
    grid = fig.add_gridspec(2,3, height_ratios=[4.2,legend_height])
    for j,(view,title) in enumerate(views):
        ax = fig.add_subplot(grid[0,j])
        draw_identity_map(ax, view, labels=labels, palette=inks, title=title,
                          size=3 if view.n_obs>8000 else 8)
        ax.set(xlim=limits[0], ylim=limits[1])
    phenotype_key(fig.add_subplot(grid[1,:]), labels, inks, columns=3)
    fig.suptitle(f"Pancreas · {panel}")
    plt.show()

## 3. Marker expression on the UMAP

Hormones, structural channels and immune identity markers share the same embedding. Each figure includes cells from donors who measured every marker in that figure. The panels keep the cohort UMAP coordinates and gray denotes unavailable measurements within that selection. Select additional measured channels with `UMAP_MARKERS`. The heatmaps below cover all identity-marker families.

In [ ]:
from phenotype_signatures import donor_marker_coverage, marker_panels

marker_coverage = donor_marker_coverage(counts, evidence, tissue="Pancreas")
unknown = set(UMAP_MARKERS).difference(marker_coverage.columns)
if unknown:
    raise ValueError(f"Choose calibrated pancreas markers from marker_coverage: {sorted(unknown)}")
positions = np.asarray(mapped.obsm["X_umap"])
lower, upper = positions.min(axis=0), positions.max(axis=0)
margin = .025 * (upper - lower)
for title, markers, donors in marker_panels(marker_coverage, UMAP_MARKERS):
    view = mapped[mapped.obs.donor.astype(str).isin(donors)].copy()
    if not view.n_obs:
        continue
    axes = sc.pl.umap(view, color=markers, layer="typing_scaled", use_raw=False, ncols=3,
                      size=4, cmap="viridis", vmin=0, vmax=1, na_color="#cecece",
                      wspace=.26, hspace=.32, show=False)
    figure_axes = axes if isinstance(axes, list) else [axes]
    fig = figure_axes[0].figure
    for axis in figure_axes:
        axis.set_xlim(lower[0]-margin[0], upper[0]+margin[0])
        axis.set_ylim(lower[1]-margin[1], upper[1]+margin[1])
    fig.suptitle(f"Pancreas · {title}", y=1.02)
    plt.show()

## 4. Shared and donor-subset proteins by cell subtype

Each row is the deepest recorded assignment. A macrophage has a positive CD68 or CD163 gate, including cells positive for both. Measured galectin-3 (M2Gal3) splits that parent into Gal3+ and Gal3− macrophages. CD4 is measured across the full panel and used for T-cell subtyping. Resolved T-cell and macrophage subtypes, competing endocrine calls and cells with insufficient lineage evidence stay visible. The primary figures show markers measured in every pancreas donor. Supplementary figures group the remaining markers by the exact donors that measured them. Assigned cell types and assignment QC have separate figures. Measured zeros count toward coverage. The panels include all typing proteins, including B3TUBB and CD56 for nerve and Podoplanin for lymphatic cells. EpCAM was excluded from the current typing workflow.

Scanpy scales each **dotplot marker column** from its lowest to highest measured subtype mean within that figure (`standard_scale="var"`). Dot area gives the fraction of measured cells with calibrated signal ≥0.10. The separate cell-type and QC heatmaps share one range for each marker within the same tissue, donor selection and donor-coverage panel. The lowest and highest measured means across the two figures map to 0 and 1. Single-marker panels show calibrated means on a numeric axis, with the same limits for cell types and QC. If an entire row falls in the highest color band, a quantitative profile shows the original calibrated means. Hatching identifies a measured marker with no variation. A gray × in the dotplot or a solid gray heatmap tile indicates missing or noncomparable measurements. Use the calibrated means in the returned table for absolute comparisons across panels. Row labels show cell and donor support. Cohort and disease-group means weight each contributing donor equally by default. Set `WEIGHTING = "cells"` to pool measured cells.

In [ ]:
WEIGHTING = "donors"  # "cells" pools the same measured cells across donors.
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame([{"Marker set": title, "Proteins": ", ".join(markers), "Donors with measurements": ", ".join(donors)}
                          for title, markers, donors in marker_panels(marker_coverage, marker_coverage.columns)]))
cohort_markers = show_profiles(counts, evidence, level=LEVEL, weighting=WEIGHTING)
display(cohort_markers.reset_index())

In [ ]:
group_markers = show_profiles(counts, evidence, level=LEVEL, disease=DISEASE, weighting=WEIGHTING)
donor_markers = show_profiles(counts, evidence, level=LEVEL, donor=DONOR)
display(donor_markers.reset_index())

### Endocrine calls with conflicting hormones

The native image audit groups the final unresolved cells by their measured competing pair and decision reason. Area gap is measured in percentage points of the full eroded interior. The two brightness columns use uncapped marker-specific fluorescence in that same interior. The named-cell comparisons below use the original cytoplasmic measurement compartment for **both** groups.

In [ ]:
for scope, selection in [("All donors", {}), (DISEASE, {"disease": DISEASE}), (f"Donor {DONOR}", {"donor": DONOR})]:
    print(scope)
    conflicts = conflict_summary(data, counts, **selection)
    if conflicts.empty:
        print("No remaining hormone conflicts.")
    else:
        display(conflicts)

### Matched hormone expression in named cells and cells with competing hormones

Each hormone column compares the unresolved pair with its corresponding named subtype in exactly the same donors. Each row gives the native competing pair, population, measured cells and matched / available donors. Blank entries are outside that comparison. A gray × indicates that the comparison has no matched measurements.

Color scales the means within each hormone column. Dot area shows the measured fraction ≥0.10. The table retains the calibrated and uncapped cytoplasmic means, matched cell counts and both donor-weighted and pooled estimates. The native-interior conflict measurements above use a different compartment.

In [ ]:
from phenotype_signatures import endocrine_evidence, matched_hormones, plot_matched_hormones

pair_measurements, named_support = endocrine_evidence(data, counts)
for scope, selection in [("All donors", {}), (DISEASE, {"disease": DISEASE}), (f"Donor {DONOR}", {"donor": DONOR})]:
    print(scope)
    matched = matched_hormones(pair_measurements, **selection)
    if matched.empty:
        print("No residual pair in this selection.")
    else:
        plot_matched_hormones(matched, scope=scope, weighting=WEIGHTING)
        display(matched.round(3))

### How much evidence supports the named endocrine groups?

The full-section census distinguishes a uniquely highest hormone from a direct endocrine Single and a missing donor/stratum refinement bound. These conditions can overlap. “Below 0.10” refers only to the display threshold in the measured defining hormone. A cell may have low fluorescence even when the bound is available, or a strong signal when the bound is unavailable. The intersection columns identify cells with both conditions. The last column reports the weakest finite signal. Each section’s denominator and route counts are shown beneath the cohort table.

In [ ]:
columns = ["named_cells", "measured", "unique_hormone_winner", "direct_Single", "no_refinement_bound",
           "below_0.10", "below_0.10_and_no_bound", "direct_Single_below_0.10_and_no_bound"]
grouped = named_support.groupby(["named_type", "defining_hormone"], observed=True)
totals = grouped[columns].sum()
totals["weakest_measured_signal"] = grouped.weakest_measured_signal.min()
display(totals)
display(named_support.sort_values(["disease", "donor", "named_type"]))

### Do neighboring protein profiles support the final cell identity?

The production graph compares each cell with its 30 nearest neighbors in the full donor protein representation. Concordance is the fraction sharing the final label. Below 0.40 means fewer than 12 of 30 share that label. A measured zero means no same-label neighbor. An unavailable graph measurement stays missing. The producer uses the donor- and stratum-specific defining-marker bound to review calls. Appearance on the two-dimensional UMAP alone does not decide a label.

The first table uses **all admitted cells** and shows outcomes for each final lineage. The second separates evaluated cells from those with an unavailable bound or another recorded outcome in the selected donor.

In [ ]:
from phenotype_signatures import neighbor_support, summarize_neighbors

refinement_qc = neighbor_support(data, counts)
lineage_qc = summarize_neighbors(refinement_qc, ["celltype"])
display(lineage_qc.rename(index=DISPLAY_NAMES).round(3))
selected_qc = summarize_neighbors(refinement_qc.loc[refinement_qc.donor.eq(DONOR)], ["celltype", "refine_state"])
display(selected_qc.round(3))

The two maps use the same saved coordinates. The right panel shows the selected donor on the cohort axes. Color comes directly from the production graph and gray marks missing concordance. The following counts use only displayed points. The selected donor lookup reports the identities, first-pass assignment and agreement for the lowest-agreement measured cells.

In [ ]:
agreement = "refinement_concordance"
if agreement not in mapped.obs:
    raise KeyError("The display sample must carry production refinement_concordance")
fig, axes = plt.subplots(1,2,figsize=(10.7,4.4),sharex=True,sharey=True)
summaries = []
for view, ax, title in [(mapped, axes[0], "All donors"),
                       (mapped[mapped.obs.donor.astype(str).eq(DONOR)].copy(), axes[1], f"Donor {DONOR}")]:
    numeric = pd.to_numeric(view.obs[agreement],errors="raise")
    measured = numeric.notna()
    if not numeric[measured].between(0,1).all():
        raise ValueError("UMAP point agreement lies outside 0 to 1")
    if len(view):
        sc.pl.umap(view, color=agreement, ax=ax, vmin=0, vmax=1, cmap="viridis",
                   na_color="#c2c5c6", size=6 if view is mapped else 14, show=False,
                   title=f"{title} · {measured.sum():,} graph-measured")
    ax.set(xlim=limits[0],ylim=limits[1],aspect="equal")
    summaries.append({"display sample":title,"coordinates":len(view),"graph measured":int(measured.sum()),
                      "low agreement (<0.40)":int(numeric.lt(.40).sum()),"zero agreement":int(numeric.eq(0).sum()),
                      "unmeasured":int(numeric.isna().sum())})
fig.suptitle("Fraction of protein neighbors with the same final label",fontsize=12)
fig.tight_layout();plt.show()
display(pd.DataFrame(summaries).set_index("display sample"))
lookup = mapped.obs.loc[mapped.obs.donor.astype(str).eq(DONOR),
                        ["celltype","first_pass","refine_state",agreement]].dropna(subset=[agreement])
lookup.index.name = "native_cell_id"
display(lookup.sort_values(agreement,kind="stable").head(20))

## 5. Explore marker co-expression

Each donor contributes 4,000 randomly selected cells. The panels show the measured cells in the disease group and selected donor. Axes use aligned protein intensity after log(1 + intensity). The scale retains differences above the ceiling used for typing.

Each point has two finite protein measurements in the same cell. The separate identity and QC figures share axes. The table reports the contributing donors for each pair.

In [ ]:
scatter_cells = sc.read_h5ad(data / "pancreas_random.h5ad")
scatter_cells.obs["subtype"] = cell_subtypes(scatter_cells.obs, "Pancreas")
from phenotype_signatures import donor_marker_coverage
coverage = donor_marker_coverage(counts, evidence, tissue="Pancreas")
if any(marker not in coverage for marker in PAIR):
    raise ValueError(f"Choose two assayed PhenoCycler markers: {PAIR}")
paired_donors = coverage.index[coverage.loc[:,list(PAIR)].all(axis=1)].astype(str)
pair = scatter_cells[scatter_cells.obs.donor.astype(str).isin(paired_donors), list(PAIR)].copy()
raw = np.asarray(pair.X)
good = np.isfinite(raw).all(axis=1) & (raw >= 0).all(axis=1)
pair = pair[good].copy()
pair.X = np.log1p(np.asarray(pair.X))
views = [(pair[pair.obs.disease.astype(str).eq(DISEASE)].copy(), DISEASE),
         (pair[pair.obs.donor.astype(str).eq(DONOR)].copy(), f"Donor {DONOR}")]
all_labels = names_in_order(set().union(*(view.obs.subtype.dropna() for view,_ in views)), "Pancreas")
pair_support = []
values = np.asarray(pair.X)
if len(values):
    lower, upper = values.min(axis=0), values.max(axis=0)
    margin = np.maximum((upper-lower)*.035, .01)
for panel, labels in category_panels(all_labels, "Pancreas"):
    legend_height = .24*np.ceil(len(labels)/3) + .3
    fig = plt.figure(figsize=(13,4.9+legend_height),layout="constrained")
    grid = fig.add_gridspec(2,2,height_ratios=[4,legend_height])
    for j,(scope,title) in enumerate(views):
        view = scope[scope.obs.subtype.isin(labels)].copy()
        donors = sorted(view.obs.donor.astype(str).unique(), key=int)
        pair_support.append({"selection":title,"panel":panel,"cells with both markers":view.n_obs,
                             "donors":len(donors),"donor IDs":", ".join(donors)})
        ax = fig.add_subplot(grid[0,j])
        if view.n_obs:
            view.obs["subtype"] = view.obs.subtype.cat.remove_unused_categories()
            view.obs["subtype"] = view.obs.subtype.cat.reorder_categories(view.obs.subtype.value_counts().index)
            sc.pl.scatter(view, x=PAIR[0], y=PAIR[1], color="subtype",
                          palette=[inks[label] for label in view.obs.subtype.cat.categories],
                          size=8 if view.n_obs<8000 else 4, marker="o", alpha=.78, ax=ax, show=False,
                          use_raw=False, legend_loc="none", frameon=True)
        else:
            ax.text(.5,.5,"No paired measurements",ha="center",va="center",transform=ax.transAxes)
        if len(values):
            ax.set(xlim=(lower[0]-margin[0],upper[0]+margin[0]),ylim=(lower[1]-margin[1],upper[1]+margin[1]))
        ax.grid(False)
        ax.set(title=f"{title} · {view.n_obs:,} cells · {len(donors)} donor(s)",
               xlabel=f"log1p({PAIR[0]})",ylabel=f"log1p({PAIR[1]})")
    phenotype_key(fig.add_subplot(grid[1,:]), labels, inks, columns=3)
    fig.suptitle(f"Marker co-expression · {panel}")
    plt.show()
display(pd.DataFrame(pair_support).set_index(["selection","panel"]))

Full donor files contain all 68 marker columns with availability and compartment metadata, both expression scales, original and updated cell types, and native coordinates. Change the field in the optional example to use other Scanpy functions.

In [ ]:
donor = sc.read_h5ad(data / "pancreas" / f"{DONOR}.h5ad", backed="r")
print(donor)
donor.obs["subtype"] = cell_subtypes(donor.obs, "Pancreas")
display(donor.obs["subtype"].value_counts().rename(index=subtype_display_name).rename("admitted cells").to_frame())
# For your own Scanpy plot, load just the desired cells/markers from the backed object:
# small = donor[:10000, ["GCG","INS","SST"]].to_memory()
# sc.pl.violin(small, ["GCG","INS","SST"], groupby="subtype", layer="typing_scaled")

Continue with [spatial analysis and donor trajectories](03_spatial_analysis.ipynb) or [group comparisons and proportions](04_group_comparisons.ipynb).